# Chapter 11: Working with Files

**MAD 2502 — Computational Mathematics with Python**

Files let information persist after Python stops running. Different formats store different kinds of information.

**Learning rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect.

## Learning goals

You should be able to work with paths, read/write text files, recognize CSV/JSON/Pickle/HDF5, explain which formats are human-readable, and choose a reasonable format for a task.

## Understand: where is Python looking?

A relative filename is interpreted from the current working directory. `pathlib.Path` gives us a readable way to inspect and construct paths.

In [ ]:
from pathlib import Path

print(Path.cwd())

## TXT: plain text

Text files are simple and human-readable. Use `with open(...)` so Python closes the file automatically.

In [ ]:
text_path = Path('mad2502_demo.txt')

with text_path.open('w', encoding='utf-8') as file:
    file.write('first line\n')
    file.write('second line\n')

with text_path.open('r', encoding='utf-8') as file:
    text = file.read()

print(text)

## Predict → Run: append vs. overwrite

Mode `'w'` starts a new file or replaces its contents. Mode `'a'` appends to the end. Predict what the file contains after the next cell.

In [ ]:
with text_path.open('a', encoding='utf-8') as file:
    file.write('third line\n')

print(text_path.read_text(encoding='utf-8'))

## CSV: rows and columns

CSV means **comma-separated values**. It is a portable text format for tables. pandas is usually the easiest tool once the data naturally form rows and columns.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'species': ['Blue Jay', 'Northern Cardinal', 'American Crow'],
    'count': [12, 18, 7],
})
csv_path = Path('mad2502_demo.csv')
df.to_csv(csv_path, index=False)

loaded = pd.read_csv(csv_path)
print(loaded)

## JSON: structured text

JSON is a language-independent text format commonly used by APIs. JSON objects resemble Python dictionaries, and JSON arrays resemble Python lists, but JSON is **not Python code**.

In [ ]:
import json

record = {
    'school': 'UF',
    'address': {'city': 'Gainesville', 'state': 'Florida'},
    'courses': ['MAD2502', 'MAD4401'],
}
json_path = Path('mad2502_demo.json')

with json_path.open('w', encoding='utf-8') as file:
    json.dump(record, file, indent=2)

with json_path.open('r', encoding='utf-8') as file:
    loaded_record = json.load(file)

print(type(loaded_record))
print(loaded_record['address']['city'])

## Pickle: Python objects

Pickle can store many Python objects conveniently, but it is Python-specific and binary rather than human-readable.

**Security rule:** only unpickle files you trust. Loading an untrusted pickle can execute malicious code.

In [ ]:
import pickle

pickle_path = Path('mad2502_demo.pkl')
values = {'a': [1, 2, 3], 'message': 'hello'}

with pickle_path.open('wb') as file:
    pickle.dump(values, file)

with pickle_path.open('rb') as file:
    restored = pickle.load(file)

print(restored)

## HDF5: hierarchical binary data

HDF5 stands for **Hierarchical Data Format, version 5**. Unlike CSV, it can store groups and datasets in a tree-like structure and is useful for large or complex scientific data. It is binary, so you do not inspect it meaningfully in a text editor.

In [ ]:
import h5py
import numpy as np

h5_path = Path('mad2502_demo.h5')
with h5py.File(h5_path, 'w') as file:
    group = file.create_group('experiment')
    group.create_dataset('measurements', data=np.array([1.2, 1.5, 1.1, 1.7]))

with h5py.File(h5_path, 'r') as file:
    print(list(file.keys()))
    print(file['experiment/measurements'][:])

## Choosing a format

| Format | Human-readable? | Good for |
|---|---|---|
| TXT | Yes | simple text, logs, small custom data |
| CSV | Yes | rectangular tables |
| JSON | Yes | nested structured data, APIs |
| Pickle | No | trusted Python-only object storage |
| HDF5 | No | large/hierarchical scientific data |

When a website offers a documented API or downloadable CSV, that is usually preferable to scraping HTML.

## Python modules are files too

A `.py` file can contain reusable Python code. In this course, `my_api_info.py` is an example of a small module used to keep a personal API key out of a notebook. **Never commit a real key to GitHub.**

In [ ]:
module_path = Path('mad2502_helper.py')
module_path.write_text('def double(x):\n    return 2 * x\n', encoding='utf-8')

import importlib.util
spec = importlib.util.spec_from_file_location('mad2502_helper', module_path)
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)
print(helper.double(8))

## Change

Add one row to the DataFrame and save the CSV again. Add one nested field to the JSON object. Reopen both files and verify that your change was saved.

## Practice

1. Explain `FileNotFoundError` in your own words.
2. Which format would you choose for a simple table shared with Excel users?
3. Which format naturally represents nested API data?
4. Why should you avoid loading an unknown pickle file?
5. What makes HDF5 different from TXT/CSV/JSON?
6. Before reading a file, predict the Python object you expect to get back.

## Connect

Capstone projects often begin with a file or API response. Understanding formats and paths makes your analysis reproducible. For common file-path and dirty-data problems, use `applied_data/04_data_rescue_kit.ipynb`.